# Diabetes Prediction System: Machine Learning Based Early-Stage Risk Prediction
### Complete Technical Project Implementation (Review I & Review II)
**Student Name:** Agarshna R  
**Register Number:** RA2411026050061  
**Department:** B.Tech Computer Science and Engineering (Artificial Intelligence & Machine Learning)  
**Institution:** SRM Institute of Science and Technology, Tiruchirappalli  
**Academic Year:** 2026–2027  

---
### Abstract
This project develops a reproducible, leakage-safe, and explainable machine learning system for early-stage diabetes risk screening using demographic and clinical symptom indicators. The raw dataset contains 520 records across 16 predictor variables and 1 binary target class. Following data validation, 269 exact duplicates were removed to prevent evaluation inflation, resulting in 251 unique records. Feature engineering created three interpretable aggregate symptom burden scores: `Symptom_Count`, `Core_Symptom_Score`, and `Associated_Symptom_Score`. Eight candidate machine learning classifiers across six experimental feature sets were benchmarked using Stratified Cross-Validation. Model predictions are interpreted using SHAP (Shapley Additive exPlanations) alongside an interactive counterfactual what-if simulation tool.

## 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pointbiserialr
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss
)
# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Matplotlib styling
plt.style.use('default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.autolayout'] = True
print('All libraries loaded successfully!')

## 2. Dataset Loading and Initial Inspection
The dataset contains 520 records and 17 columns (16 predictors + `class`).

In [ ]:
raw_df = pd.read_csv('../data/diabetes_data_upload.csv')
print('Raw Dataset Dimensions:', raw_df.shape)
raw_df.head()

## 3. Data Preprocessing and Leakage Control
- Missing value audit
- Duplicate row identification and removal (269 exact duplicates)
- Binary categorical encoding (Gender: Female=0, Male=1; Symptoms: No=0, Yes=1; Target: Negative=0, Positive=1)
- Age outlier screening using standard Interquartile Range (IQR)

In [ ]:
# Check missing values
print('Missing Values Count per Column:')
print(raw_df.isnull().sum())

# Check and remove exact duplicates
num_duplicates = raw_df.duplicated().sum()
print(f'\nExact Duplicate Rows Detected: {num_duplicates}')
clean_df = raw_df.drop_duplicates().copy()
print(f'Clean Deduplicated Records: {clean_df.shape[0]}')

# Encode categorical columns
binary_symptoms = [
    'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness',
    'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching',
    'Irritability', 'delayed healing', 'partial paresis',
    'muscle stiffness', 'Alopecia', 'Obesity'
]

clean_df['Gender'] = clean_df['Gender'].map({'Female': 0, 'Male': 1})
clean_df['class'] = clean_df['class'].map({'Negative': 0, 'Positive': 1})
for s in binary_symptoms:
    clean_df[s] = clean_df[s].map({'No': 0, 'Yes': 1})

# Age Outlier Analysis
q1 = clean_df['Age'].quantile(0.25)
q3 = clean_df['Age'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
flagged_outliers = clean_df[(clean_df['Age'] < lower_bound) | (clean_df['Age'] > upper_bound)]

print(f'Age IQR: {iqr:.2f} | Lower Bound: {lower_bound:.2f} | Upper Bound: {upper_bound:.2f}')
print(f'Flagged Observations: {len(flagged_outliers)} (Age 90 - clinically retained)')

## 4. Exploratory Data Analysis (EDA)
Visualizing target distribution, age distribution, symptom prevalence, and correlation matrix.

In [ ]:
# Figure 1: Target Class Distribution
counts = clean_df['class'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(['Negative', 'Positive'], [counts[0], counts[1]], color='#1f77b4', width=0.45)
ax.set_title('Figure 1: Target Class Distribution')
ax.set_ylabel('Number of records')
plt.show()

# Figure 2: Age Distribution
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.hist(clean_df['Age'], bins=15, color='#1f77b4', edgecolor='white')
ax.set_title('Figure 2: Distribution of Participant Age')
ax.set_xlabel('Age (Years)')
ax.set_ylabel('Frequency')
plt.show()

## 5. Feature Engineering
Creating clinically grounded aggregate symptom scores:
1. `Symptom_Count`: Sum of all 14 symptom indicators.
2. `Core_Symptom_Score`: Sum of core osmotic symptoms (Polyuria, Polydipsia, sudden weight loss, Polyphagia, weakness).
3. `Associated_Symptom_Score`: Sum of secondary associated symptoms.

In [ ]:
core_symptoms = ['Polyuria', 'Polydipsia', 'sudden weight loss', 'Polyphagia', 'weakness']
assoc_symptoms = [
    'Genital thrush', 'visual blurring', 'Itching', 'Irritability',
    'delayed healing', 'partial paresis', 'muscle stiffness', 'Alopecia', 'Obesity'
]

fe_df = clean_df.copy()
fe_df['Symptom_Count'] = fe_df[binary_symptoms].sum(axis=1)
fe_df['Core_Symptom_Score'] = fe_df[core_symptoms].sum(axis=1)
fe_df['Associated_Symptom_Score'] = fe_df[assoc_symptoms].sum(axis=1)

for col in ['Symptom_Count', 'Core_Symptom_Score', 'Associated_Symptom_Score']:
    s = fe_df[col]
    print(f'{col}: min={s.min()}, max={s.max()}, mean={s.mean():.2f}, median={s.median():.2f}')

## 6. Feature Relevance Analysis (Cramer's V, Mutual Information, RF Gini)

In [ ]:
# Compute Cramer's V with Yates' correction
assoc_list = []
n = len(clean_df)
for s in binary_symptoms + ['Gender']:
    cm = pd.crosstab(clean_df[s], clean_df['class'])
    chi2, p, _, _ = chi2_contingency(cm, correction=True)
    v = np.sqrt(chi2 / n)
    assoc_list.append({'Feature': s, 'Association': v, 'p-value': p})

df_assoc = pd.DataFrame(assoc_list).sort_values(by='Association', ascending=False)
print('Leading Feature Associations (Table 7.1):')
print(df_assoc.head(8))

## 7. Machine Learning Modeling & Stratified Cross-Validation
Evaluating 8 candidate classifiers across the 6 planned feature sets.

In [ ]:
from src.modeling import evaluate_models_cross_validation, train_and_save_best_model
cv_results = evaluate_models_cross_validation(fe_df, n_splits=5)
cv_results.to_csv('../saved_models/cv_model_comparison.csv', index=False)
print('Top 10 Model Configurations by ROC-AUC:')
cv_results.sort_values(by='ROC_AUC', ascending=False)[['Feature_Set', 'Model', 'Accuracy', 'Recall', 'Specificity', 'F1_Score', 'ROC_AUC', 'Brier_Score']].head(10)

## 8. Explainable AI: SHAP Local & Global Explanations and Counterfactual What-If Simulation

In [ ]:
from src.explainability import DiabetesExplainer
explainer = DiabetesExplainer('../saved_models/best_model.pkl', '../saved_models/scaler.pkl', '../saved_models/model_feature_names.pkl')

# Define a test patient
patient = {
    'Age': 50, 'Gender': 1, 'Polyuria': 1, 'Polydipsia': 1,
    'sudden weight loss': 1, 'weakness': 1, 'Polyphagia': 0,
    'Genital thrush': 0, 'visual blurring': 0, 'Itching': 1,
    'Irritability': 1, 'delayed healing': 1, 'partial paresis': 0,
    'muscle stiffness': 0, 'Alopecia': 0, 'Obesity': 1
}
pred, prob = explainer.predict_patient_risk(patient)
risk_label = 'High Risk' if pred == 1 else 'Low Risk'
print(f'Predicted Patient Risk: {prob*100:.1f}% ({risk_label})')

# Local Explanation
explanation = explainer.explain_patient(patient)
print('\nLocal Feature Attributions:')
print(explanation['attributions'].head(6))

# What-If Counterfactual: Resolve Polyuria & Polydipsia
whatif = explainer.simulate_what_if(patient, {'Polyuria': 0, 'Polydipsia': 0})
print('\nWhat-If Counterfactual Simulation:')
base_p = whatif['baseline_probability'] * 100
mod_p = whatif['modified_probability'] * 100
delta_p = whatif['delta_percent']
print(f'Baseline Prob: {base_p:.1f}% -> Modified Prob: {mod_p:.1f}%')
print(f'Risk Probability Delta: {delta_p:.1f}%')

## 9. Conclusion & Review Summary
- Successfully reproduced and completed the end-to-end diabetes prediction framework.
- Data cleaning resolved 269 duplicates without contradictory label patterns.
- Feature engineering added 3 interpretable scores that improve clinical representation.
- Stratified cross-validation verified that Random Forest and SVM achieve superior discrimination (ROC-AUC ~0.97).
- SHAP local attributions and What-If counterfactual simulation provide clinicians and patients with transparent, actionable risk factors.